# Standard 13.3M proxy: 32 shared trials

Select the RTX PRO 6000 GPU in Colab, run the cells in order, and approve the Google Drive mount when prompted.


In [ ]:
# Experiment storage
USE_GOOGLE_DRIVE = True

from pathlib import Path
import os

default_workspace = "/content/bmr" if Path("/content").is_dir() else "~/bmr-workspace"
WORKSPACE = Path(os.environ.get("BMR_WORKSPACE", default_workspace)).expanduser().resolve()
EXPERIMENT = "standard_vs_gated_160m_tinyllama"
DATA = WORKSPACE / "data" / "fineweb_edu_tinyllama"
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    my_drive = Path("/content/drive/MyDrive")
    if not my_drive.is_dir():
        raise RuntimeError("Mount Google Drive before continuing.")
    storage = my_drive / "bmr"
else:
    storage = WORKSPACE
RUNS = storage / "runs" / EXPERIMENT
SAVED_DATA = storage / "data" / "fineweb_edu_tinyllama"
print("Experiment outputs:", RUNS)


In [ ]:
# Run controls
MICROBATCH_SIZE = 4  # Sequences per GPU microbatch. Choose a divisor of 128.
RUN_SMOKE = True
RUN_FULL = False
# RUN_SMOKE trains one full trial (~267M tokens, 20 validations).
# RUN_FULL runs the 32-trial search and skips completed trials, including trial 000.


In [ ]:
# Set up experiment runtime
"""Standard-library setup embedded in the notebooks so it can run before the checkout exists."""

import fcntl
import json
import re
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

GITHUB_REPO_URL = "https://github.com/seungjun-green/base-model-research"


def _checkout(workspace, runs):
    source_file = runs / "source.json"
    if source_file.exists():
        source = json.loads(source_file.read_text())
        if source["url"].rstrip("/").removesuffix(".git").lower() != GITHUB_REPO_URL.lower():
            raise ValueError(
                f"{runs} is already pinned to a different repository. Use a new EXPERIMENT name."
            )
        commit = source["commit"]
    else:
        head = subprocess.check_output(
            ["git", "ls-remote", "--exit-code", GITHUB_REPO_URL, "HEAD"], text=True
        )
        commit = head.split()[0]
        source = {"url": GITHUB_REPO_URL, "commit": commit}
    if not re.fullmatch(r"[0-9a-f]{40}", commit):
        raise ValueError(f"Invalid Git commit for this experiment: {commit!r}")

    repo = workspace / "repos" / commit
    if not repo.exists():
        repo.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--", GITHUB_REPO_URL, repo], check=True)
    if subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip():
        raise RuntimeError(f"The runtime checkout has local edits: {repo}")
    if subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip() != commit:
        subprocess.run(["git", "fetch", "origin", commit], cwd=repo, check=True)
        subprocess.run(["git", "checkout", "--detach", commit], cwd=repo, check=True)
    if not source_file.exists():
        temporary = source_file.with_suffix(".tmp")
        temporary.write_text(json.dumps(source, indent=2) + "\n")
        temporary.replace(source_file)
    return repo


def setup_colab(workspace, runs, *, benchmark):
    runs.mkdir(parents=True, exist_ok=True)
    with (runs / ".setup.lock").open("a") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX)
        repo = _checkout(workspace, runs)
        try:
            torch_version = version("torch")
        except PackageNotFoundError as error:
            raise RuntimeError("Select a Colab runtime with PyTorch installed and rerun setup.") from error

        # Keep the runtime's PyTorch build, including its CUDA version suffix.
        extras = "data,eval" if benchmark else "data"
        print(f"Using {sys.executable} with PyTorch {torch_version}", flush=True)
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                f"torch=={torch_version}",
                "-e",
                f"{repo}[{extras}]",
            ],
            check=True,
        )
    return repo

REPO = setup_colab(WORKSPACE, RUNS, benchmark=False)

sys.path.insert(0, str(REPO / "src"))
from base_model_research.progress import run_with_progress

"""Proxy notebook helpers, embedded so they work with an existing experiment's pinned trainer."""

import json
import shutil
from pathlib import Path

import torch
from tqdm.auto import tqdm

from base_model_research.config import ModelConfig, TrainConfig, dump_json
from base_model_research.data import TokenCorpus, file_sha256
from base_model_research.diagnostics import DEFAULT_HP
from base_model_research.models import ResearchLM, parameter_count
from base_model_research.storage import run_lock


def stage_proxy_data(source, data, training_targets):
    """Stage the training prefix (including lookahead) and every validation shard."""
    manifest = source / "manifest.json"
    if not manifest.is_file():
        raise FileNotFoundError(f"Upload the prepared FineWeb-Edu folder to {source} before training.")
    local_manifest = data / "manifest.json"
    if source == data or (local_manifest.is_file() and local_manifest.read_bytes() == manifest.read_bytes()):
        print(f"Using the prepared local dataset at {data}.", flush=True)
        return data

    train = TokenCorpus(source, "train")
    validation = TokenCorpus(source, "validation")
    if train.total_tokens < training_targets + 1:
        raise ValueError(f"This proxy needs {training_targets + 1:,} training tokens including lookahead.")
    local = data.parent / "proxy_shards" / train.fingerprint
    with run_lock(local, blocking=True):
        required = []
        for corpus, limit in ((train, training_targets + 1), (validation, validation.total_tokens)):
            for start, entry in zip(corpus.starts, corpus.entries):
                name = entry["path"]
                if Path(name).name != name or name in ("", ".", ".."):
                    raise ValueError(f"Expected a shard filename: {name!r}")
                if start < limit:
                    required.append(entry)
                elif not (local / name).exists():
                    # Pinned trainers stat every shard listed in the original manifest.
                    # Only shards beyond this proxy's training budget are links to Drive.
                    (local / name).symlink_to(source / name)

        print(f"Preparing {sum(e['tokens'] * 4 for e in required) / 1e9:.2f} GB at {local}", flush=True)
        for entry in tqdm(required, desc="Copying or verifying proxy shards", unit="shard"):
            target = local / entry["path"]
            if not target.is_symlink() and target.is_file() and file_sha256(target) == entry["sha256"]:
                continue
            temporary = target.with_suffix(".copying")
            try:
                shutil.copyfile(source / entry["path"], temporary)
                if file_sha256(temporary) != entry["sha256"]:
                    raise ValueError(f"Shard checksum mismatch: {entry['path']}")
                temporary.replace(target)
            finally:
                temporary.unlink(missing_ok=True)
        # Keep the full corpus identity for saved trials and the main-model transfer.
        dump_json(local / "manifest.json", train.manifest)
    return local


def proxy_training_targets(repo, family):
    model = ModelConfig.load(repo / "configs" / "models" / f"{family}_proxy.yaml")
    config = TrainConfig.load(repo / "configs" / "training.yaml")
    with torch.device("meta"):
        parameters = parameter_count(ResearchLM(model, DEFAULT_HP))
    return parameters * config.tokens_per_parameter


def proxy_sweep_args(repo, runs, family, microbatch_size, local):
    model_name = f"{family}_proxy"
    model_path = repo / "configs" / "models" / f"{model_name}.yaml"
    if type(microbatch_size) is not int:
        raise ValueError("MICROBATCH_SIZE must be a positive integer that divides the global batch size.")
    config = TrainConfig.load(repo / "configs" / "training.yaml", microbatch_size=microbatch_size)
    protocol = runs / "proxy" / family / "protocol.json"
    settings_path = protocol if protocol.is_file() else runs / f"{model_name}_profile.json"
    settings = json.loads(settings_path.read_text())["training"] if settings_path.is_file() else {}
    if settings_path == protocol and settings["microbatch_size"] != microbatch_size:
        raise ValueError(
            f"This {family} sweep uses MICROBATCH_SIZE = {settings['microbatch_size']}. "
            f"Use that value to continue, or set a new EXPERIMENT name to start a sweep at {microbatch_size}."
        )
    accumulation = config.global_batch_size // microbatch_size
    print(f"{model_name}: {accumulation} microbatches of {microbatch_size} sequences per update.", flush=True)
    args = [
        "--model",
        str(model_path),
        "--data",
        str(local),
        "--output",
        str(runs / "proxy"),
        "--microbatch-size",
        str(microbatch_size),
        "--progress-events",
    ]
    if settings.get("activation_checkpointing", config.activation_checkpointing):
        args.append("--activation-checkpointing")
    return args


In [ ]:
# Copy proxy data from Google Drive to /content
PROXY_DATA = stage_proxy_data(SAVED_DATA, DATA, proxy_training_targets(REPO, "standard"))
print(f"Proxy data ready at {PROXY_DATA}.")


In [ ]:
FAMILY_RUNS = RUNS / "proxy" / "standard"

def run_proxy(*, first_only=False):
    sweep_args = proxy_sweep_args(REPO, RUNS, "standard", MICROBATCH_SIZE, PROXY_DATA)
    options = ["--end", 1] if first_only else []
    print("Starting the standard proxy trainer…", flush=True)
    run_with_progress([sys.executable, "-u", "-m", "base_model_research.cli", "sweep",
                       *sweep_args, *map(str, options)], cwd=REPO)


In [ ]:
# Smoke run · train trial 000 on 20 tokens per parameter
if RUN_SMOKE:
    run_proxy(first_only=True)
    result = json.loads((FAMILY_RUNS / "trial_000" / "result.json").read_text())
    print(f"Trial 000 is {result['status']} at step {result['step']}.")
else:
    print("Smoke run disabled.")


In [ ]:
# Full run · all 32 trials
if RUN_FULL:
    run_proxy()
    winner = FAMILY_RUNS / "best_hparams.json"
    if winner.exists():
        best = json.loads(winner.read_text())
        print(f"Winner: {best['trial_id']} · validation loss {best['validation_loss']:.4f}")
        print(best["hyperparameters"])
    else:
        print("No winner yet. See", FAMILY_RUNS / "results.csv")
else:
    print("Full run disabled.")
